# Notebook 03 — Pratique Python : Analyse depuis PostgreSQL
**Base :** `cyclistic_db` | **Table :** `cyclistic_clean`  
**Objectif :** Reproduire et approfondir l'analyse Cyclistic en Python pur —  
pandas pour les calculs, matplotlib/seaborn pour les visualisations,  
le tout alimenté directement depuis PostgreSQL.

---
## Plan
1. Connexion & chargement des données
2. Exploration rapide du DataFrame
3. Analyse 1 — Statistiques descriptives
4. Analyse 2 — Comportement par jour de la semaine
5. Analyse 3 — Évolution mensuelle et saisonnière
6. Analyse 4 — Répartition par heure
7. Analyse 5 — Types de vélos
8. Analyse 6 — Top 10 stations casual
9. Visualisations rapides avec matplotlib
10. Exercices pratiques


---
## 1. Connexion & chargement

On charge `cyclistic_clean` depuis PostgreSQL directement dans un DataFrame pandas.  
> **Note :** 3,88M lignes → quelques secondes de chargement. C'est normal.

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
sns.set_theme(style='whitegrid', palette='muted')
PALETTE = {'member': '#1A73E8', 'casual': '#E84D0E'}

# Connexion
conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='cyclistic_db',
    user='postgres', password='postgres'
)
print('Connecte.')

In [ ]:
# Chargement complet en DataFrame
print('Chargement en cours...')
cur = conn.cursor()
cur.execute('SELECT * FROM cyclistic_clean')
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
cur.close()

df = pd.DataFrame(rows, columns=cols)

# Conversions de types
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at']   = pd.to_datetime(df['ended_at'])
df['ride_length'] = df['ride_length'].astype(float)

print(f'Charge : {len(df):,} lignes x {len(df.columns)} colonnes')

---
## 2. Exploration rapide

In [ ]:
# Types de colonnes et valeurs manquantes
info = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null': df.isna().sum(),
    'null_%': (df.isna().sum() / len(df) * 100).round(1)
})
info

In [ ]:
# Aperçu des premières lignes
df.head(3)

---
## 3. Statistiques descriptives

`.groupby()` + `.agg()` : regrouper par une colonne et calculer plusieurs statistiques en une fois.

In [ ]:
desc = df.groupby('member_casual')['ride_length'].agg(
    total_rides='count',
    mean_min='mean',
    median_min='median',
    max_min='max',
    std_min='std'
).round(2).reset_index()

# Ajouter le pourcentage du total
desc['pct'] = (desc['total_rides'] / desc['total_rides'].sum() * 100).round(1)

desc

In [ ]:
# Ratio casual / member
m = desc[desc['member_casual']=='member']['mean_min'].values[0]
c = desc[desc['member_casual']=='casual']['mean_min'].values[0]
print(f'Duree moyenne membre : {m:.1f} min')
print(f'Duree moyenne casual : {c:.1f} min')
print(f'Ratio casual/member  : {c/m:.2f}x')

---
## 4. Comportement par jour de la semaine

In [ ]:
DOW_LABELS = {1:'Dim', 2:'Lun', 3:'Mar', 4:'Mer', 5:'Jeu', 6:'Ven', 7:'Sam'}

dow = df.groupby(['member_casual', 'day_of_week']).agg(
    total_rides=('ride_id', 'count'),
    avg_length=('ride_length', 'mean')
).round(2).reset_index()

dow['day_name'] = dow['day_of_week'].map(DOW_LABELS)
dow

In [ ]:
# Visualisation : trajets par jour
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title, ylabel in zip(
    axes,
    ['total_rides', 'avg_length'],
    ['Nombre de trajets par jour', 'Duree moyenne par jour (min)'],
    ['Nombre de trajets', 'Duree moyenne (min)']
):
    for user, color in PALETTE.items():
        sub = dow[dow['member_casual'] == user]
        ax.bar(
            [x + (0.2 if user == 'casual' else -0.2) for x in sub['day_of_week']],
            sub[metric],
            width=0.4, color=color, label=user.capitalize(), alpha=0.85
        )
    ax.set_xticks(range(1, 8))
    ax.set_xticklabels(list(DOW_LABELS.values()))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

---
## 5. Évolution mensuelle et saisonnière

In [ ]:
monthly = df.groupby(['member_casual', 'year', 'month']).size().reset_index(name='total_rides')
monthly['periode'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)
monthly = monthly.sort_values(['member_casual', 'year', 'month'])
monthly.head(6)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for user, color in PALETTE.items():
    sub = monthly[monthly['member_casual'] == user]
    ax.plot(sub['periode'], sub['total_rides'], marker='o',
            color=color, label=user.capitalize(), linewidth=2, markersize=5)

ax.set_title('Evolution mensuelle des trajets (2019-2020)', fontsize=13, fontweight='bold')
ax.set_ylabel('Nombre de trajets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Répartition saisonnière en % par type
seasonal = df.groupby(['member_casual', 'season']).size().reset_index(name='total_rides')
seasonal['pct'] = seasonal.groupby('member_casual')['total_rides'].transform(
    lambda x: (x / x.sum() * 100).round(1)
)
seasonal.pivot(index='season', columns='member_casual', values='pct')

---
## 6. Répartition par heure de départ

In [ ]:
hourly = df.groupby(['member_casual', 'hour_of_day']).size().reset_index(name='total_rides')

fig, ax = plt.subplots(figsize=(12, 5))
for user, color in PALETTE.items():
    sub = hourly[hourly['member_casual'] == user]
    ax.plot(sub['hour_of_day'], sub['total_rides'],
            color=color, label=user.capitalize(), linewidth=2.5, marker='o', markersize=4)

ax.set_title('Trajets par heure de depart', fontsize=13, fontweight='bold')
ax.set_xlabel('Heure de la journee')
ax.set_ylabel('Nombre de trajets')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Annoter les pics
for h, label in [(8, '8h\n(rush)'), (17, '17h\n(rush)')]:
    sub_m = hourly[(hourly['member_casual']=='member') & (hourly['hour_of_day']==h)]
    if not sub_m.empty:
        ax.annotate(label, xy=(h, sub_m['total_rides'].values[0]),
                    xytext=(h+0.5, sub_m['total_rides'].values[0]+5000),
                    fontsize=9, color=PALETTE['member'])

ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Types de vélos

In [ ]:
bikes = df.groupby(['member_casual', 'rideable_type']).size().reset_index(name='total_rides')
bikes['pct'] = bikes.groupby('member_casual')['total_rides'].transform(
    lambda x: (x / x.sum() * 100).round(1)
)
bikes

---
## 8. Top 10 stations casual

In [ ]:
top10 = (
    df[(df['member_casual'] == 'casual') & (df['start_station_name'].notna())]
    .groupby('start_station_name').size()
    .reset_index(name='total_rides')
    .sort_values('total_rides', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['start_station_name'][::-1], top10['total_rides'][::-1],
               color=PALETTE['casual'], alpha=0.85)
ax.set_title('Top 10 stations de depart - Casual riders', fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre de trajets')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Ajouter les valeurs sur les barres
for bar, val in zip(bars, top10['total_rides'][::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 9. Techniques pandas utiles à retenir

Ce tableau résume les opérations les plus utilisées dans ce notebook :

In [ ]:
recap = pd.DataFrame({
    'Operation': [
        'groupby + agg',
        'groupby + size',
        'transform',
        'pivot',
        'merge / join',
        'sort_values',
        'head(n)',
        'isna().sum()',
        'astype()',
        'str.contains()',
    ],
    'Usage': [
        'Regrouper et calculer plusieurs statistiques',
        'Compter les lignes par groupe',
        'Calculer un % au sein du groupe (sans perdre les lignes)',
        'Transformer lignes en colonnes',
        'Joindre deux DataFrames (comme SQL JOIN)',
        'Trier par une colonne',
        'Garder les N premieres lignes',
        'Compter les valeurs manquantes',
        'Convertir le type d une colonne',
        'Filtrer par correspondance de texte',
    ],
    'Exemple': [
        "df.groupby('col').agg(n=('id','count'))",
        "df.groupby('col').size()",
        "df.groupby('g')['v'].transform(lambda x: x/x.sum())",
        "df.pivot(index='a', columns='b', values='c')",
        "pd.merge(df1, df2, on='key')",
        "df.sort_values('col', ascending=False)",
        "df.head(10)",
        "df.isna().sum()",
        "df['col'].astype(float)",
        "df['col'].str.contains('pattern')",
    ]
})
recap

---
## 10. Exercices pratiques

### Exercice 1
Calcule la **durée totale des trajets en heures** (pas en minutes) par saison et par type d'utilisateur.  
Quel groupe cumule le plus d'heures en été ?

In [ ]:
# Ton code ici
# Indice : df.groupby([...])['ride_length'].sum() / 60

### Exercice 2
Trouve les **5 trajets les plus longs** (en minutes) pour les casual riders.  
Affiche : `started_at`, `ended_at`, `ride_length`, `start_station_name`.

In [ ]:
# Ton code ici
# Indice : df[condition].sort_values(...).head(5)

### Exercice 3
Trace un graphique en barres montrant le **nombre de trajets par heure**,  
uniquement pour les **samedis**, membres vs casual côte à côte.

In [ ]:
# Ton code ici
# Indice : filtrer df[df['day_of_week'] == 7], puis groupby + plot

### Exercice 4 (avancé)
Crée une fonction `top_stations(user_type, n=10)` qui retourne les N stations  
les plus fréquentées pour un type d'utilisateur donné, avec le nombre et le % de trajets.

In [ ]:
# Ton code ici
def top_stations(user_type, n=10):
    pass

# top_stations('casual', 10)
# top_stations('member', 5)

In [ ]:
conn.close()
print('Connexion fermee.')